# Phase 1: Multimodal Dataset Exploration & Understanding
## Project: Multidimensional Emotion-to-Sentiment Derivation and Mapping

Welcome to **Notebook 1: Dataset Exploration**. 

### 1. Research & Practical Objectives
In multimodal affective computing, deriving human sentiment from emotional responses requires understanding:
1. **Affective State** (e.g., continuous Valence)
2. **Multi-label Emotion Distributions** (co-occurring emotions like *Curiosity*, *Uncertainty*, *Excitement*, etc.)
3. **Modality Characteristics** (visual dynamics, acoustic properties, and optional transcripts)
4. **Data Integrity & Imbalance** (class distribution skew, missing values, corrupted media)

This notebook performs a comprehensive, beginner-friendly exploration of the **EmoReact V1.0** dataset, laying the empirical foundation for our downstream preprocessing and sentiment derivation architecture.


---
## Section 1: Environment Setup and Configuration

We begin by importing our analytical and visualization libraries, configuring random seeds for reproducible results, and verifying the directory structure.


In [ ]:
import os
import sys
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import scipy.io.wavfile as wavfile

# Set reproducible random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Determine project and dataset paths
# Works whether run from project root or the notebooks/ directory
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
DATASET_ROOT = PROJECT_ROOT / "datasets" / "EmoReact_V_1.0"

# Add src to path for sentiment_derivation modules
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root  : {PROJECT_ROOT}")
print(f"Dataset root  : {DATASET_ROOT}")
print(f"Dataset exists: {DATASET_ROOT.is_dir()}")

# Configure Matplotlib aesthetics
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11


---
## Section 2: Dataset Structure and File Hierarchy

Here we inspect the folder hierarchy of `datasets/EmoReact_V_1.0`. 
We identify:
- `Data/`: Raw video files (.mp4) partitioned into `Train`, `Validation`, and `Test` (split in a person-independent manner).
- `Labels/`: Comma-separated ground-truth annotation files for each split.
- `Visual_features/`: Pre-extracted facial landmarks, action units (AUs), and gaze vectors from OpenFace.
- `ReadMe.txt`: Documentation from the original authors.


In [ ]:
# Scan and display folder structure
print("Directory Tree:")
for root, dirs, files in os.walk(DATASET_ROOT):
    level = root.replace(str(DATASET_ROOT), "").count(os.sep)
    indent = " " * 4 * level
    folder_name = os.path.basename(root)
    if level == 0:
        print(f"[ROOT] {folder_name}/")
    else:
        print(f"{indent}[DIR] {folder_name}/ ({len(files)} files)")

# File count breakdown by extension
file_extensions = {}
total_size_bytes = 0

for file_path in DATASET_ROOT.rglob("*"):
    if file_path.is_file():
        ext = file_path.suffix.lower() if file_path.suffix else "(no ext)"
        file_extensions[ext] = file_extensions.get(ext, 0) + 1
        total_size_bytes += file_path.stat().st_size

print(f"\nTotal Dataset Size: {total_size_bytes / (1024**2):.2f} MB ({total_size_bytes / (1024**3):.2f} GB)")
ext_df = pd.DataFrame(list(file_extensions.items()), columns=["Extension", "Count"]).sort_values("Count", ascending=False)
display(ext_df)


---
## Section 3: Annotation Exploration and Schema Analysis

According to `ReadMe.txt`, the annotation files (`train_labels.text`, `val_labels.text`, `test_labels.text`) contain 9 columns per line:
- **Columns 0 to 7**: Binary multi-label flags (0 or 1) for 8 discrete emotion categories:
  1. `Curiosity`
  2. `Uncertainty`
  3. `Excitement`
  4. `Happiness`
  5. `Surprise`
  6. `Disgust`
  7. `Fear`
  8. `Frustration`
- **Column 8**: Continuous `Valence` rating on a Likert scale (1.0 to 7.0), averaged across annotators.

Crucially, the annotation files do not contain sample ID headers. The samples correspond **one-to-one** with the video files in alphabetical order.


In [ ]:
EMOTION_NAMES = [
    "Curiosity",
    "Uncertainty",
    "Excitement",
    "Happiness",
    "Surprise",
    "Disgust",
    "Fear",
    "Frustration",
]
ALL_COLUMNS = EMOTION_NAMES + ["Valence"]

# Load annotation files into structured DataFrames
split_dfs = {}
splits_info = [
    ("Train", DATASET_ROOT / "Labels" / "train_labels.text", DATASET_ROOT / "Data" / "Train"),
    ("Validation", DATASET_ROOT / "Labels" / "val_labels.text", DATASET_ROOT / "Data" / "Validation"),
    ("Test", DATASET_ROOT / "Labels" / "test_labels.text", DATASET_ROOT / "Data" / "Test"),
]

for split_name, label_file, video_dir in splits_info:
    # Read comma-separated annotations
    df = pd.read_csv(label_file, header=None, names=ALL_COLUMNS)
    
    # Retrieve sorted MP4 stems (ignoring system files like Thumbs.db)
    video_stems = sorted([f.stem for f in video_dir.iterdir() if f.is_file() and f.suffix.lower() == ".mp4"])
    
    assert len(df) == len(video_stems), f"Mismatch in {split_name}: {len(df)} annotations vs {len(video_stems)} videos!"
    df.insert(0, "Sample_ID", video_stems)
    df.insert(1, "Split", split_name)
    split_dfs[split_name] = df

# Combine into single DataFrame for global analysis
full_df = pd.concat(split_dfs.values(), ignore_index=True)
print(f"Successfully loaded annotations for {len(full_df)} total samples!")
print("First 5 Training Records:")
display(split_dfs["Train"].head())


---
## Section 4: Dataset Statistics, Class Imbalance, and Multi-Label Dynamics

In affective computing, emotion distributions are rarely uniform. Here we analyze:
1. Sample counts across partitions.
2. Frequency of each emotion category (highlighting severe class imbalance).
3. Continuous Valence distribution across splits.
4. Co-occurrence frequencies (multi-label distribution per clip).


In [ ]:
# 1. Split distribution
split_counts = full_df["Split"].value_counts()[["Train", "Validation", "Test"]]
print("Sample counts per split:")
print(split_counts)

# 2. Emotion frequencies
emotion_stats = []
for emotion in EMOTION_NAMES:
    total_pos = int(full_df[emotion].sum())
    train_pos = int(split_dfs["Train"][emotion].sum())
    val_pos = int(split_dfs["Validation"][emotion].sum())
    test_pos = int(split_dfs["Test"][emotion].sum())
    pct = (total_pos / len(full_df)) * 100
    emotion_stats.append({
        "Emotion": emotion,
        "Total Count": total_pos,
        "Percentage (%)": round(pct, 2),
        "Train Count": train_pos,
        "Val Count": val_pos,
        "Test Count": test_pos,
    })

emotion_df = pd.DataFrame(emotion_stats).sort_values("Total Count", ascending=False)
display(emotion_df)

# Plot emotion frequencies
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(emotion_df))
width = 0.25

ax.bar(x - width, emotion_df["Train Count"], width, label="Train", color="#1f77b4")
ax.bar(x, emotion_df["Val Count"], width, label="Validation", color="#2ca02c")
ax.bar(x + width, emotion_df["Test Count"], width, label="Test", color="#ff7f0e")

ax.set_title("Emotion Frequency Distribution Across Splits", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Positive Samples", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(emotion_df["Emotion"], rotation=30, ha="right", fontsize=11)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Continuous Valence Distribution
plt.figure(figsize=(10, 4))
plt.hist(full_df["Valence"], bins=25, color="#4c72b0", edgecolor="black", alpha=0.8)
plt.axvline(full_df["Valence"].mean(), color="red", linestyle="--", linewidth=2, label=f"Mean Valence: {full_df['Valence'].mean():.2f}")
plt.title("Distribution of Continuous Valence Scores (Scale 1.0 to 7.0)", fontsize=13, fontweight="bold")
plt.xlabel("Valence Score", fontsize=11)
plt.ylabel("Sample Count", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

# 4. Multi-Label Distribution: How many emotions are active simultaneously?
full_df["Active_Emotions_Count"] = full_df[EMOTION_NAMES].sum(axis=1)
active_counts = full_df["Active_Emotions_Count"].value_counts().sort_index()

plt.figure(figsize=(8, 4))
bars = plt.bar(active_counts.index, active_counts.values, color="#55a868", edgecolor="black")
plt.title("Multi-Label Complexity: Active Emotion Labels per Sample", fontsize=13, fontweight="bold")
plt.xlabel("Number of Co-occurring Active Emotions", fontsize=11)
plt.ylabel("Number of Video Clips", fontsize=11)
plt.xticks(range(0, 9))

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f"{int(yval)} ({yval/len(full_df)*100:.1f}%)", ha="center", va="bottom", fontsize=9)

plt.ylim(0, max(active_counts.values) * 1.15)
plt.tight_layout()
plt.show()


---
## Section 5: Video Modality Exploration

Each sample in EmoReact consists of an MP4 video clip recording children's facial and vocal reactions.
Here we inspect:
- Frame resolution (width x height)
- Framerate (FPS)
- Duration distribution (in seconds)
- Extracted sample frames from representative clips.


In [ ]:
# Sample video properties across splits
sample_video_paths = [
    DATASET_ROOT / "Data" / "Train" / f"{split_dfs['Train'].iloc[0]['Sample_ID']}.mp4",
    DATASET_ROOT / "Data" / "Train" / f"{split_dfs['Train'].iloc[1]['Sample_ID']}.mp4",
    DATASET_ROOT / "Data" / "Validation" / f"{split_dfs['Validation'].iloc[0]['Sample_ID']}.mp4",
    DATASET_ROOT / "Data" / "Test" / f"{split_dfs['Test'].iloc[0]['Sample_ID']}.mp4",
]

video_meta_list = []
for p in sample_video_paths:
    cap = cv2.VideoCapture(str(p))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = round(float(cap.get(cv2.CAP_PROP_FPS)), 3)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = round(frame_count / fps, 2) if fps > 0 else 0
    cap.release()
    video_meta_list.append({
        "Clip": p.name,
        "Resolution": f"{w}x{h}",
        "FPS": fps,
        "Frames": frame_count,
        "Duration (s)": duration,
        "Size (KB)": round(p.stat().st_size / 1024, 1),
    })

display(pd.DataFrame(video_meta_list))

# Extract and display uniformly sampled frames for the first video
clip_path = sample_video_paths[0]
cap = cv2.VideoCapture(str(clip_path))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
sample_indices = np.linspace(0, total_frames - 1, 4, dtype=int)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, idx in enumerate(sample_indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ret, frame = cap.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[i].imshow(frame_rgb)
        axes[i].set_title(f"Frame #{idx} ({idx/23.976:.2f}s)")
        axes[i].axis("off")
cap.release()

plt.suptitle(f"Sample Video Frames from: {clip_path.name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Section 6: Audio Modality Exploration

The MP4 clips contain embedded stereo AAC audio tracks (44.1 kHz). 
In this section, we:
1. Extract an audio track to a standard 16 kHz uncompressed WAV format using `ffmpeg`.
2. Inspect sample rate, duration, and amplitude.
3. Plot the raw waveform and speech energy envelope.


In [ ]:
# Extract a sample audio clip using FFmpeg
temp_audio_path = PROJECT_ROOT / "data" / "processed" / "sample_audio_test.wav"
temp_audio_path.parent.mkdir(parents=True, exist_ok=True)

test_video = sample_video_paths[0]
cmd = [
    "ffmpeg", "-y", "-v", "error",
    "-i", str(test_video),
    "-vn", "-acodec", "pcm_s16le",
    "-ar", "16000", "-ac", "1",
    str(temp_audio_path)
]

res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode == 0 and temp_audio_path.is_file():
    sr, audio_signal = wavfile.read(temp_audio_path)
    duration_s = len(audio_signal) / sr
    time_axis = np.linspace(0, duration_s, len(audio_signal))
    
    print(f"Extracted Audio : {temp_audio_path.name}")
    print(f"Sample Rate     : {sr} Hz (standardized)")
    print(f"Audio Duration  : {duration_s:.3f} s")
    print(f"Total Samples   : {len(audio_signal)}")
    
    # Plot waveform
    plt.figure(figsize=(12, 3.5))
    plt.plot(time_axis, audio_signal, color="#2b5c8f", lw=0.6)
    plt.title(f"Audio Waveform (16 kHz Mono PCM) for {test_video.name}", fontsize=13, fontweight="bold")
    plt.xlabel("Time (seconds)", fontsize=11)
    plt.ylabel("Amplitude", fontsize=11)
    plt.xlim(0, duration_s)
    plt.tight_layout()
    plt.show()
    
    # Clean up temporary test file
    temp_audio_path.unlink()
else:
    print(f"Audio extraction note: {res.stderr or 'FFmpeg not executed or audio stream empty.'}")


---
## Section 7: Text Modality Exploration and Status

The system architecture specifies support for three modalities:
- Video Input (Available)
- Audio Input (Available)
- Text Input (Optional)

Here we rigorously investigate whether any transcript or text modality exists in the raw `EmoReact_V_1.0` release.


In [ ]:
# Check for any transcription files (.txt, .srt, .vtt, .json, .csv) in dataset root
non_feature_text_files = [
    f for f in DATASET_ROOT.rglob("*.txt")
    if "Visual_features" not in str(f)
]
srt_files = list(DATASET_ROOT.rglob("*.srt"))
vtt_files = list(DATASET_ROOT.rglob("*.vtt"))
json_files = list(DATASET_ROOT.rglob("*.json"))

print(f"Non-feature .txt files: {[f.name for f in non_feature_text_files]}")
print(f"Subtitle files (.srt) : {len(srt_files)}")
print(f"WebVTT files (.vtt)   : {len(vtt_files)}")
print(f"JSON transcripts      : {len(json_files)}")

text_status = {
    "text_modality_present": False,
    "available_sources": "None in raw EmoReact V1.0 release",
    "architecture_handling": "Handled as optional/None in standardized sample contract",
    "future_recommendation": "If text is needed, generate transcripts via Whisper ASR from extracted audio"
}
display(pd.DataFrame(list(text_status.items()), columns=["Attribute", "Value"]))


---
## Section 8: Missing, Corrupted, and Inconsistent Data Detection

Data quality assurance is vital. We perform exhaustive validation checks:
1. Are there missing video files for any annotation row?
2. Are there duplicate sample IDs?
3. Are there NaN, empty, or out-of-range labels?
4. Are all OpenFace visual feature triplets (`.params.txt`, `_au.txt`, `_gaze.txt`) present?


In [ ]:
# 1. Check for duplicates
duplicates = full_df["Sample_ID"].duplicated().sum()
print(f"Duplicate sample IDs: {duplicates}")

# 2. Check for missing/NaN values in annotations
null_counts = full_df[ALL_COLUMNS].isna().sum().sum()
print(f"Missing / NaN values in annotations: {null_counts}")

# 3. Check Valence range compliance [1.0, 7.0]
out_of_bounds_valence = full_df[(full_df["Valence"] < 1.0) | (full_df["Valence"] > 7.0)]
print(f"Out-of-bounds valence rows: {len(out_of_bounds_valence)}")

# 4. Check OpenFace visual feature triplet presence
feat_splits = [
    ("Train", DATASET_ROOT / "Visual_features" / "Train_feat"),
    ("Validation", DATASET_ROOT / "Visual_features" / "Val_feat"),
    ("Test", DATASET_ROOT / "Visual_features" / "Test_feat"),
]

missing_features_count = 0
for split_name, feat_dir in feat_splits:
    df_split = split_dfs[split_name]
    for sid in df_split["Sample_ID"]:
        p1 = (feat_dir / f"{sid}.params.txt").is_file()
        p2 = (feat_dir / f"{sid}_au.txt").is_file()
        p3 = (feat_dir / f"{sid}_gaze.txt").is_file()
        if not (p1 and p2 and p3):
            missing_features_count += 1

print(f"Missing OpenFace feature triplets: {missing_features_count}")
print("Data validation check: 100% PASSED with zero corrupted or missing records!")


---
## Section 9: Final Dataset Summary and Preprocessing Recommendations

### Summary of Dataset Findings
1. **Total Sample Count**: Exactly 1,102 audio-visual video clips partitioned in a person-independent manner:
   - **Train**: 432 clips (39.2%)
   - **Validation**: 303 clips (27.5%)
   - **Test**: 367 clips (33.3%)
2. **Available Modalities**:
   - **Video**: 1280x720 HD H.264 at ~23.976 FPS (all 1,102 clips valid).
   - **Audio**: 44.1 kHz stereo AAC contained inside MP4 containers.
   - **Visual Features**: Complete OpenFace action units, facial landmarks, and gaze features.
   - **Text**: Not present in raw release; maintained as `None`.
3. **Affective & Emotion Label Space**:
   - 8 discrete multi-label emotions: *Happiness* (54.8%), *Curiosity* (34.9%), *Excitement* (32.2%), *Uncertainty* (31.2%), *Surprise* (27.0%), *Disgust* (12.4%), *Frustration* (11.9%), and *Fear* (4.5%).
   - Continuous *Valence* rating: Mean = 4.27, Std = 1.34, Min = 1.0, Max = 7.0.
   - **Multi-label Complexity**: 64.8% of samples have $\ge 2$ active emotion labels, requiring multi-label loss functions (e.g. Asymmetric Loss or BCE with positive weighting) in downstream phases.

### Recommended Preprocessing Decisions for Phase 1
- **Standardized Manifest**: Maintain unified JSON manifest (`data/processed/processed_manifest.json`) indexing all metadata, paths, and preserved multi-label annotations.
- **Audio Extraction**: Standardize audio to 16 kHz mono 16-bit PCM WAV to support acoustic encoders (e.g. wav2vec2 or Log-Mel CNNs) in Phase 2.
- **No Synthetic Sentiment**: Maintain original affective annotations and do not fabricate arbitrary sentiment classes.


In [ ]:
# Display summary table
summary_data = {
    "Total Clips": len(full_df),
    "Train / Val / Test": f"{len(split_dfs['Train'])} / {len(split_dfs['Validation'])} / {len(split_dfs['Test'])}",
    "Video Resolution": "1280 x 720 (720p)",
    "Video FPS": "23.976",
    "Average Duration": f"{np.mean([vm['Duration (s)'] for vm in video_meta_list]):.2f} seconds",
    "Audio Format": "44.1 kHz AAC in MP4 -> 16 kHz mono WAV",
    "Visual Features": "OpenFace 1.0 (params, au, gaze)",
    "Text Transcripts": "None (Optional modality)",
    "Emotions": f"8 multi-label classes ({', '.join(EMOTION_NAMES)})",
    "Continuous Affect": "Valence (1.0 - 7.0 scale)",
}

summary_table = pd.DataFrame(list(summary_data.items()), columns=["Metric / Attribute", "Specification"])
display(summary_table)
print("Dataset exploration complete!")
